In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from datetime import datetime

# Lecture Bronze
df = spark.table("pharma_catalog.bronze.bronze_products")
display(df)

In [0]:
# COMMAND ----------
# Doublons
df_dedup = df.dropDuplicates(["product_id"])
doublons = df.count() - df_dedup.count()
print(f"🔁 Doublons détectés : {doublons}")

# COMMAND ----------
df_quality = df_dedup.withColumn(
    "rule_failed",
    F.when(F.col("product_id").isNull(), "null_product_id")
    .when(F.col("product_name").isNull(), "null_product_name")
    .when(F.col("unit_price").isNull(), "null_unit_price")
    .when(F.col("unit_price") < 0, "prix_negatif")
    .when(F.col("category").isNull(), "null_category")
    .when(F.col("manufacturer").isNull(), "null_manufacturer")
    .when(F.col("status").isNull(), "null_status")
    .otherwise(None)
)

# Lignes OK
passed = df_quality.filter(F.col("rule_failed").isNull()).drop("rule_failed")

# Lignes rejetées
failed = df_quality.filter(F.col("rule_failed").isNotNull())

print(f"✅ Lignes OK : {passed.count()}")
print(f"❌ Lignes rejetées : {failed.count()}")

# COMMAND ----------
# Quarantine
if failed.count() > 0:
    df_quarantine = failed.withColumn("source_table", F.lit("bronze_products")) \
                           .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))

    df_quarantine.write.mode("append").saveAsTable("pharma_catalog.silver_quarantine.quarantine")
    print(f"✅ {failed.count()} ligne(s) envoyée(s) en quarantine")
else:
    print("✅ Aucune ligne rejetée — quarantine vide")

In [0]:
# COMMAND ----------
# Voir la ligne rejetée
display(failed)

In [0]:
# COMMAND ----------
# Envoi des lignes rejetées vers la Quarantine

df_quarantine = failed.withColumn("source_table", F.lit("bronze_products")) \
                       .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))

df_quarantine.write.mode("append").saveAsTable("pharma_catalog.silver_quarantine.quarantine_products")

print(f" {failed.count()} ligne(s) envoyée(s) en quarantine")

In [0]:
# COMMAND ----------
# Envoi des lignes propres vers Silver

passed.write.mode("overwrite").saveAsTable("pharma_catalog.silver.silver_products")

print(f"✅ {passed.count()} ligne(s) chargée(s) dans silver_products")

In [0]:
# COMMAND ----------
# Vérification Silver
display(spark.table("pharma_catalog.silver.silver_products"))

# COMMAND ----------
# Vérification Quarantine
display(spark.table("pharma_catalog.silver_quarantine.quarantine_products"))